[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qubvel/segmentation_models.pytorch/blob/main/examples/camvid_segmentation_multiclass.ipynb)

# Install package

In [ ]:
%%capture
!pip install --upgrade lightning albumentations huggingface_hub[hf_xet]-q
!pip install segmentation-models-pytorch -q
!pip install monai -q

# Loading data

raw data will have images shape 1024, 1024

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as BaseDataset
import segmentation_models_pytorch as smp

import cv2
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download

import albumentations as A
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [ ]:
root_dir = os.getcwd()
data_zip_path = hf_hub_download(
    repo_id = "thanglexuan/murincells",
    filename = "data.zip",
    repo_type = "dataset",
    token = secret_value_0
)
data_dir = os.path.join(root_dir, "data")
os.makedirs(data_dir, exist_ok=True)
with zipfile.ZipFile(data_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_dir)
    
images_dir = os.path.join(data_dir, "images")
masks_dir = os.path.join(data_dir, "masks")
image_ls = sorted(os.listdir(images_dir))
mask_ls = sorted(os.listdir(masks_dir))

x_train_dir, x_temp_dir, y_train_dir, y_temp_dir = train_test_split(
    image_ls, mask_ls, test_size=0.3, random_state=42
)
x_val_dir, x_test_dir, y_val_dir, y_test_dir = train_test_split(
    x_temp_dir, y_temp_dir, test_size=0.5, random_state=42
)
print("train size:{}".format(len(x_train_dir)))
print("val size:{}".format(len(x_val_dir)))
print("test size:{}".format(len(x_test_dir)))

In [ ]:
def check_files_names_validation(mask_dir, num_classes=int):
    for fname in os.listdir(mask_dir):
        mask_path = os.path.join(mask_dir, fname)
        mask = cv2.imread(mask_path, 0)  # grayscale
        if mask is None:
            continue
        max_val = mask.max()
        if max_val >= num_classes:
            print(f"⚠️ File {fname} có nhãn max = {max_val}, unique = {np.unique(mask)}")
            
mask_dir = os.path.join(data_dir, "masks_grayscale")
num_classes = 6
check_files_names_validation(mask_dir, num_classes)
print(f"All files are validation")

# Dataloader

In [ ]:
class Dataset(BaseDataset):
    CLASSES = [
        "Background",
        "Macrophage/Monocyte",
        "Neutrophil",
        "Eosinophil",
        "Lymphocyte",
        "Unknown cell/Debris"
    ]

    def __init__(self, data_dir, images_dir, masks_dir, classes=None, augmentation=None):
        self.data_dir = data_dir
        self.images_fps = images_dir
        self.masks_fps = masks_dir

        # Always map background ('Background') to 0
        self.background_class = self.CLASSES.index("Background")

        # If specific classes are provided, map them dynamically
        if classes:
            self.class_values = [self.CLASSES.index(cls.lower()) for cls in classes]
        else:
            self.class_values = list(range(len(self.CLASSES)))  # Default to all classes
            
        # Create a remapping dictionary: class value in dataset -> new index (0, 1, 2, ...)
        # Background will always be 0, other classes will be remapped starting from 1.
        self.class_map = {self.background_class: 0}
        self.class_map.update(
            {
                v: i
                for i, v in enumerate(self.class_values)
                if v != self.background_class
            }
        )
        self.augmentation = augmentation

    def __getitem__(self, i):
        # Read the image
        image_path = os.path.join(self.data_dir, "images", self.images_fps[i])
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

        # Read the mask in grayscale mode
        mask_path = os.path.join(self.data_dir, "masks_grayscale", self.masks_fps[i])
        mask = cv2.imread(mask_path, 0)

        # Create a blank mask to remap the class values
        mask_remap = np.zeros_like(mask)

        # Remap the mask according to the dynamically created class map
        for class_value, new_value in self.class_map.items():
            mask_remap[mask == class_value] = new_value

        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask_remap)
            image, mask_remap = sample["image"], sample["mask"]

        image = image.transpose(2, 0, 1)
        return image, mask_remap

    def __len__(self):
        return len(self.images_fps)

In [ ]:
def visualize(**images):
    """Plot images in one row."""
    n = len(images)
    plt.figure(figsize=(10, 5))
    for i, (name, image) in enumerate(images.items()):
        plt.subplot(1, n, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.title(" ".join(name.split("_")).title())

        # If it's an image, plot it as RGB
        if name == "image":
            # Convert CHW to HWC for plotting
            image = image.transpose(1, 2, 0)
            plt.imshow(image)
        else:
            plt.imshow(image, cmap="tab20")
    plt.show()

In [ ]:
dataset = Dataset(data_dir=data_dir, images_dir=x_train_dir, masks_dir=y_train_dir)
import random
image, mask = dataset[random.randint(0, len(dataset) - 1)]
print(f"Mask shape: {mask.shape}")
visualize(image=image, mask=mask)

In [ ]:
def get_training_augmentation():
    train_transform = [
        # Áp dụng cho cả image + mask
        A.HorizontalFlip(p=0.5),
        A.Affine(    scale=[0.5, 1],
                        translate_percent=[-0.05, 0.05],
                        rotate=[-45, 45],
                        shear=[-15, 15],
                        interpolation=cv2.INTER_LINEAR,
                        mask_interpolation=cv2.INTER_NEAREST,
                        fit_output=False,
                        keep_ratio=False,
                        rotate_method="ellipse",
                        balanced_scale=True,
                        border_mode=cv2.BORDER_CONSTANT,
                        fill=0,
                        fill_mask=0
        ),
        A.PadIfNeeded(min_height=256, min_width=256, p=1),
        A.RandomCrop(height=256, width=256, p=1),

        # Chỉ áp dụng cho ảnh
        A.OneOf([
            A.CLAHE(p=1),
            A.RandomBrightnessContrast(p=1),
            A.RandomGamma(p=1),
        ], p=0.9),
        A.OneOf([
            A.Sharpen(p=1),
            A.Blur(blur_limit=3, p=1),
            A.MotionBlur(blur_limit=3, p=1),
        ], p=0.9),
        A.OneOf([
            A.RandomBrightnessContrast(p=1),
            A.HueSaturationValue(p=1),
        ], p=0.9),
    ]
    return A.Compose(train_transform)


def get_validation_augmentation():
    valid_transform = [
        A.PadIfNeeded(min_height=256, min_width=256)
                      ]
    return A.Compose(valid_transform)

In [ ]:
# Visualize resulted augmented images and masks
augmented_dataset = Dataset(data_dir=data_dir,
    images_dir=x_train_dir,
    masks_dir=y_train_dir,
    augmentation=get_training_augmentation(),
)

# Visualizing the same image with different random transforms
for i in range(3):
    image, mask = augmented_dataset[3]
    print(f"Mask shape: {mask.shape}")
    print(np.unique(mask))
    visualize(image=image, mask=mask)

In [ ]:
train_dataset = Dataset(
    data_dir=data_dir,
    images_dir=x_train_dir,
    masks_dir=y_train_dir,
    augmentation=get_training_augmentation(),
)

valid_dataset = Dataset(
    data_dir=data_dir,
    images_dir=x_val_dir,
    masks_dir=y_val_dir,
    augmentation=get_validation_augmentation(),
)

test_dataset = Dataset(
    data_dir=data_dir,
    images_dir=x_test_dir,
    masks_dir=y_test_dir,
    augmentation=get_validation_augmentation(),
)

# Change to > 0 if not on Windows machine
BATCH_SIZE = 8
NUM_WORKERS = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Create model and train

In [ ]:
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torch
from torch.optim import lr_scheduler
# Torchmetrics
from torchmetrics.classification import Precision, Recall, F1Score, Accuracy
from torchmetrics.segmentation import DiceScore, MeanIoU, GeneralizedDiceScore

class MurineCellModel(pl.LightningModule):
    def __init__(self, arch, encoder_name, in_channels, num_classes, **kwargs):
        super().__init__()
        self.model = smp.create_model(
            arch,
            encoder_name=encoder_name,
            in_channels=in_channels,
            classes=num_classes,
            **kwargs,
        )
        self.num_classes = num_classes

        # Preprocessing parameters for image normalization
        params = smp.encoders.get_preprocessing_params(encoder_name)
        self.register_buffer("std", torch.tensor(params["std"]).view(1, 3, 1, 1))
        self.register_buffer("mean", torch.tensor(params["mean"]).view(1, 3, 1, 1))

        # Loss function for multi-class segmentation
        self.loss_fn = smp.losses.DiceLoss(smp.losses.MULTICLASS_MODE, from_logits=True)

        self.train_metrics = self._init_metrics()
        self.valid_metrics = self._init_metrics()
        self.test_metrics = self._init_metrics()

        # Step metrics tracking
        self.training_step_outputs = []
        self.validation_step_outputs = []
        self.test_step_outputs = []
        
    def _init_metrics(self):
        metrics = torch.nn.ModuleDict({
            "gds": GeneralizedDiceScore(num_classes=self.num_classes, include_background=True, per_class=False),
            "dice": DiceScore(num_classes=self.num_classes, include_background=True),
            "miou": MeanIoU(num_classes=self.num_classes, include_background=True, per_class=False),
            "precision": Precision(num_classes=self.num_classes, average="micro", task="multiclass"),
            "recall": Recall(num_classes=self.num_classes, average="micro", task="multiclass"),
            "f1": F1Score(num_classes=self.num_classes, average="micro", task="multiclass"),
            "acc": Accuracy(num_classes=self.num_classes, average="micro", task="multiclass"),
        })
        return metrics
        
    def forward(self, image):
        # Normalize image
        image = (image - self.mean) / self.std
        mask = self.model(image)
        return mask

    def shared_step(self, batch, stage):
        image, mask = batch

        # Ensure that image dimensions are correct
        assert image.ndim == 4  # [batch_size, channels, H, W]

        # Ensure the mask is a long (index) tensor
        mask = mask.long()

        # Mask shape
        assert mask.ndim == 3  # [batch_size, H, W]

        # Predict mask logits
        logits_mask = self.forward(image)

        assert (
            logits_mask.shape[1] == self.num_classes
        )  # [batch_size, num_classes, H, W]

        # Ensure the logits mask is contiguous
        logits_mask = logits_mask.contiguous()

        # Compute loss using multi-class Dice loss (pass original mask, not one-hot encoded)
        loss = self.loss_fn(logits_mask, mask)

        # Apply softmax to get probabilities for multi-class segmentation
        prob_mask = logits_mask.softmax(dim=1)

        # Convert probabilities to predicted class labels
        pred_mask = prob_mask.argmax(dim=1)
        
        # Get predictions for metrics (detached, no gradients needed)
        # with torch.no_grad():
        
        # Compute metrics using predictions
        metrics = getattr(self, f"{stage}_metrics")
        out_metrics = {name: metric(pred_mask, mask) for name, metric in metrics.items()}
        
        out_metrics["loss"] = loss
        return out_metrics

    def shared_epoch_end(self, outputs, stage):
        metrics = {}
        for key in outputs[0].keys():
            metrics[f"{stage}_{key}"] = torch.stack([x[key] for x in outputs]).mean()
        self.log_dict(metrics, prog_bar=True, sync_dist=True)
        

    def training_step(self, batch, batch_idx):
        train_loss_info = self.shared_step(batch, "train")
        self.training_step_outputs.append(train_loss_info)
        return train_loss_info

    def on_train_epoch_end(self):
        self.shared_epoch_end(self.training_step_outputs, "train")
        self.training_step_outputs.clear()

    def validation_step(self, batch, batch_idx):
        valid_loss_info = self.shared_step(batch, "valid")
        self.validation_step_outputs.append(valid_loss_info)
        return valid_loss_info

    def on_validation_epoch_end(self):
        self.shared_epoch_end(self.validation_step_outputs, "valid")
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        test_loss_info = self.shared_step(batch, "test")
        self.test_step_outputs.append(test_loss_info)
        return test_loss_info

    def on_test_epoch_end(self):
        self.shared_epoch_end(self.test_step_outputs, "test")
        self.test_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=2e-4)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
            },
        }

In [ ]:
# Some training hyperparameters
EPOCHS = 5
# Always include the background as a class
OUT_CLASSES = len(train_dataset.CLASSES)
print(OUT_CLASSES)

Unet_model = MurineCellModel("Unet", "resnext50_32x4d", in_channels=3, num_classes=OUT_CLASSES)

# Training

In [ ]:
import torch, gc
from pytorch_lightning.loggers import CSVLogger

gc.collect()
torch.cuda.empty_cache()

csv_logger = CSVLogger("logs", name="unet")

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu",
    devices=1,
    logger=csv_logger
)

trainer.fit(
    Unet_model,
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader,
)

# Validation and test metrics

In [ ]:
# run validation dataset
valid_metrics = trainer.validate(Unet_model, dataloaders=valid_loader, verbose=False)
print(valid_metrics)

In [ ]:
# run test dataset
test_metrics = trainer.test(Unet_model, dataloaders=test_loader, verbose=False)
print(test_metrics)

# Result visualization

In [ ]:
import torch
import matplotlib.pyplot as plt

def predict_and_plot(model, dataloader, device=None, n_samples=5):
    """
    Run inference on a batch from dataloader and visualize predictions.

    Args:
        model: trained PyTorch/Lightning model
        dataloader: DataLoader (val/test)
        device: torch.device (cpu/cuda)
        n_samples: number of samples to plot
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model.eval()

    # Get one batch
    images, masks = next(iter(dataloader))
    images, masks = images.to(device), masks.to(device)

    with torch.inference_mode():
        logits = model(images)
        pr_masks = logits.softmax(dim=1).argmax(dim=1)

    # Plot
    for idx, (image, gt_mask, pr_mask) in enumerate(zip(images, masks, pr_masks)):
        if idx >= n_samples:
            break

        plt.figure(figsize=(12, 6))

        # Original Image
        plt.subplot(1, 3, 1)
        plt.imshow(image.cpu().numpy().transpose(1, 2, 0))
        plt.title("Image")
        plt.axis("off")

        # Ground Truth Mask
        plt.subplot(1, 3, 2)
        plt.imshow(gt_mask.cpu().numpy(), cmap="tab20")
        plt.title("Ground truth")
        plt.axis("off")

        # Predicted Mask
        plt.subplot(1, 3, 3)
        plt.imshow(pr_mask.cpu().numpy(), cmap="tab20")
        plt.title("Prediction")
        plt.axis("off")

        plt.show()


In [ ]:
predict_and_plot(Unet_model, valid_loader, n_samples=5)